# انسان_ان_دی_لوپ: پیش_عملی دروازے، خطرے کی درجہ بندی، اور آڈٹ لاگنگ

اس سبق کی README انسانی_ان_دی_لوپ کا تعارف ایک مختصر اقتباس کے ساتھ کراتی ہے جو صارف سے `APPROVE` یا `REJECT` پوچھتی ہے جب ایجنٹ پہلے ہی ایک جواب دے چکا ہو۔ یہ پیٹرن ایک اچھا آغاز ہے، لیکن پروڈکشن HITL تنفیذات میں عام طور پر تین اضافی حصے درکار ہوتے ہیں:

1. ایک **پیش_عملی دروازہ** جو **ایجنٹ کے خطرناک قدم اٹھانے سے پہلے** چلے، تاکہ لاگت، ناقابل واپسی پن، اور تاخیر پر قابو پایا جا سکے۔
2. **خطرے کی درجہ بندی**، تاکہ کم خطرہ والے اقدامات خودکار طریقے سے اجرا ہوں، درمیانے خطرہ والے اقدامات کو مجموعی طور پر منظوری ملے، اور صرف زیادہ خطرہ والے اقدامات انسانی مداخلت پر رکے۔
3. ایک **آڈٹ لاگ اور نظرثانی کا چکر**، تاکہ ہر دروازہ فیصلہ JSONL کی صورت میں ریکارڈ ہو، اور رد کرنے پر ایجنٹ کو محض `Revising...` چھاپنے کے بجائے منظم وجہ کے ساتھ دوبارہ فعال کیا جائے۔

یہ نوٹ بُک انہی بنیادی اجزاء پر مبنی ہے جو `06-system-message-framework.ipynb` میں استعمال ہوئے ہیں۔ یہ مکمل طور پر `DEMO_MODE = True` میں چلتا ہے (جہاں کوئی متعامل ان پٹ درکار نہیں) یا جب `DEMO_MODE = False` ہو تو اصل `input()` پرامپٹس کے ساتھ۔ نوٹ کریں: DEMO_MODE میں تیسرے مقصد کی کوشش اسکپٹ کی گئی ہے تاکہ چکر کی کارکردگی مکمل طور پر دیکھی جا سکے۔ حقیقی نظرثانی سے چلنے والی دوبارہ درجہ بندی کے لیے `DEMO_MODE = False` اور ایک آپریٹر کی ضرورت ہے۔

**دائرہ کار سے باہر (دیگر اسباق میں شامل):** تصدیق اور رسائی کنٹرول (سبق 06 README خطرہ 2)، ٹول کال مڈل ویئر (سبق 14 MAF میں تفصیل)، ملٹی ایجنٹ مباحثے کے پیٹرنز۔

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## پیٹرن 1: پیش کاری گیٹ

README کا HITL اسنیپٹ پہلے ایجنٹ کو کال کرتا ہے، پھر صارف سے آؤٹ پٹ کی منظوری مانگتا ہے۔ یہ ایک **بعد از عمل** کا سلسلہ ہے۔ ایجنٹ پہلے ہی عمل کر چکا ہوتا ہے، اس لیے LLM کال کا خرچ پہلے ہی ادا ہو چکا ہوتا ہے، اور کوئی بھی ضمنی اثر (ایمیل بھیجی گئی، ڈیٹا بیس کی قطار لکھی گئی، تبصرہ پوسٹ کیا گیا) پہلے ہی ہو چکا ہوتا ہے۔

ایک **پیش کاری** سلسلہ گیٹ کو اس سے پہلے داخل کرتا ہے کہ ایجنٹ خطرناک قدم چلائے۔ ایجنٹ عمل تجویز کرتا ہے، گیٹ فیصلہ کرتا ہے کہ آیا اسے انجام دیا جائے، اور صرف منظوری پر ضمنی اثر واقع ہوتا ہے۔

| پہلو | بعد از عمل منظوری (README اسنیپٹ) | پیش کاری گیٹ (اس نوٹ بک میں) |
|---|---|---|
| منظوری کب چلتی ہے؟ | جب ایجنٹ نے آؤٹ پٹ پیدا کر لیا ہو | جب کوئی بھی ضمنی اثر عمل میں نہیں آیا ہو |
| رد کرنے پر LLM خرچ | پہلے ہی ادا ہو چکا ہے | صرف تجویز کے لیے ادا ہوتا ہے، عمل کے لیے نہیں |
| ناقابل واپسی ضمنی اثرات | ممکن ہیں (عمل پہلے ہی ہو چکا ہے) | روکا گیا ہے |
| آڈٹ کی وضاحت | منظوری ایک پرنٹ بیان ہے | منظوری ایک JSON ریکارڈ ہے جس میں ٹائم اسٹیمپ، عمل، وجہ شامل ہے |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## پیٹرن 2: رسک ٹائرنگ

ہر عمل کو انسانی منظوری کی ضرورت نہیں ہوتی۔ ایک عوامی API کے خلاف صرف پڑھنے والا لوک اپ ایک صارف کو ای میل بھیجنے سے مختلف خطرات رکھتا ہے۔ دونوں کو ایک جیسا سمجھنا آپریٹر کی توجہ ضائع کرتا ہے اور ایجنٹ کو سست کر دیتا ہے۔

ایک سادہ 3-سطحی ماڈل:

| سطح | مثالیں | منظوری کا عمل |
|---|---|---|
| `کم` (صرف پڑھنے کے لئے) | نالج بیس تلاش کرنا، پرواز کے اختیارات دیکھنا، عوامی ویب پیج حاصل کرنا | خودکار عمل، آڈٹ کے لیے لاگ کیا گیا |
| `درمیانہ` (سستا تبدیلی) | نتیجہ کیش کرنا، کاؤنٹر بڑھانا، یاد دہانی شیڈول کرنا | خودکار عمل، لیکن روزانہ جائزے کے لیے بیچ میں رکھنا |
| `زیادہ` (بیرونی یا ناقابل واپسی) | ای میل بھیجنا، کارڈ چارج کرنا، عوامی چینل پر پوسٹ کرنا | انسانی منظوری پر روکنا |

یہ ایک سطح بندی ہے۔ پروڈکشن سسٹمز اکثر زیادہ تفصیلی سطحیں استعمال کرتے ہیں (مثلاً، AWS IAM پرمیشن لیولز، کردار پر مبنی رسائی کی سطحیں)۔ نیچے دیا گیا 3-سطحی ورژن ایک ایجنٹ کے لیے سب سے چھوٹا مفید ورژن ہے جو صرف پڑھنے اور اثرانداز ہونے والے عمل کو مکس کرتا ہے۔

نیچے دیا گیا کلاسفائر کی ورڈ ہیروسٹکس استعمال کرتا ہے تاکہ ڈیمو متعین اور سستا رہے۔ پروڈکشن سسٹم میں آپ اس کی جگہ سیکھا ہوا کلاسفائر یا پالیسی انجن لگائیں گے۔

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## پیٹرن ۳: آڈٹ لاگ اور نظرثانی کا لوپ

`print("Response approved.")` کوئی آڈٹ لاگ نہیں ہے۔ اعتماد کے لیے، ہر گیٹ کا فیصلہ ایک منظم واقعے کے طور پر ریکارڈ ہونا چاہیے جسے آپ بعد میں پوچھ گچھ، دوبارہ چلانے، یا واقعہ کے جائزے سے منسلک کر سکیں۔

دو حصے:

1. **صرف اضافہ کرنے والا JSONL۔** ہر فیصلے کے لیے ایک لائن، جس میں وقت، عمل، سطح، فیصلہ، وجہ شامل ہو۔ تلاش کرنا آسان، بعد میں حقیقی لاگ اسٹور میں بھیجنا آسان۔
2. **رد پر نظرثانی کا لوپ۔** جب گیٹ `deny` واپس کرتا ہے، تو ایجنٹ خود کو رد کرنے کی وجہ کے ساتھ دوبارہ دعوت دیتا ہے، تاکہ اگلی تجویز مسئلہ سے بچ سکے۔

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## اضافی وسائل

کئی دوسرے عوامی منصوبے ان HITL پیٹرنز کی مختلف اقسام کو نافذ کرتے ہیں۔ اپنے اسٹیک کے لیے بہترین اپروچ تلاش کرنے کے لیے طریقہ کار کا موازنہ کریں:

- **LangChain** human-in-the-loop ٹول ریپرز ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ایسے ڈراپ-ان ٹول ریپرز جو انسانی ان پٹ کے لیے عمل کو روک دیتے ہیں۔
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ نے اسے دوبارہ ترتیب دیا): ایک ایجنٹ کردار استعمال کرتا ہے جو کثیر ایجنٹ بات چیت میں خاص طور پر انسان کی نمائندگی کرتا ہے۔
- **Semantic Kernel** فنکشن فلٹرز ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): مڈل ویئر جو ہر فنکشن کال کے ارد گرد چلتا ہے، نگران منطق کے لیے موزوں ہے۔

ہر منصوبہ تین ذیلی پیٹرنز کو مختلف طریقے سے ہینڈل کرتا ہے: LangChain انہیں ٹولز کے طور پر ریپ کرتا ہے، AutoGen ایجنٹ کردار استعمال کرتا ہے، Semantic Kernel مڈل ویئر فلٹرز استعمال کرتا ہے۔ اپنا ایجنٹ ڈیزائن منتخب کرنے سے پہلے ایک یا دو امپلیمنٹیشنز کو ابتدا سے انتہا تک پڑھیں۔

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
